In [1]:
!pip install -q kagglehub timm scikit-learn

import os
import torch
import torch.nn as nn
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Використовується пристрій:", device)

import kagglehub

path = kagglehub.dataset_download("gpiosenka/cards-image-datasetclassification")
print("Шлях до завантаженого датасету:", path)

Використовується пристрій: cuda


100%|██████████| 385M/385M [00:04<00:00, 92.4MB/s]

Extracting files...


Шлях до завантаженого датасету: /root/.cache/kagglehub/datasets/gpiosenka/cards-image-datasetclassification/versions/2


In [2]:
import glob

train_dir = os.path.join(path, "train")
test_dir = os.path.join(path, "test")

print("Папка train:", train_dir)
print("Папка test:", test_dir)

from torchvision.datasets import ImageFolder

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

train_dataset = ImageFolder(root=train_dir, transform=transform)
test_dataset = ImageFolder(root=test_dir, transform=transform)

print(f"Кількість зображень у train: {len(train_dataset)}")
print(f"Кількість зображень у test: {len(test_dataset)}")
print(f"Кількість класів (типів карток): {len(train_dataset.classes)}")

Папка train: /root/.cache/kagglehub/datasets/gpiosenka/cards-image-datasetclassification/versions/2/train
Папка test: /root/.cache/kagglehub/datasets/gpiosenka/cards-image-datasetclassification/versions/2/test
Кількість зображень у train: 7624
Кількість зображень у test: 265
Кількість класів (типів карток): 53


In [4]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)

base_model = timm.create_model('efficientnet_b0', pretrained=True)

features_extractor = nn.Sequential(*list(base_model.children())[:-1])
features_extractor = features_extractor.to(device)
features_extractor.eval()

train_features = []
train_labels = []

print("Витягуємо ознаки для тренувальних зображень...")
with torch.no_grad():
    for images, labels in train_loader:
        images = images.to(device)

        feats = features_extractor(images)

        feats = torch.flatten(feats, 1)

        train_features.append(feats.cpu().numpy())
        train_labels.append(labels.numpy())

X_train = np.concatenate(train_features, axis=0)
y_train = np.concatenate(train_labels, axis=0)

print("Форма масиву X_train (фічі):", X_train.shape)
print("Форма масиву y_train (мітки):", y_train.shape)

Витягуємо ознаки для тренувальних зображень...
Форма масиву X_train (фічі): (7624, 1280)
Форма масиву y_train (мітки): (7624,)


In [6]:
from sklearn.metrics.pairwise import cosine_similarity

# 1. Витягуємо ознаки для тестових зображень
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

test_features = []
test_labels = []

print("Витягуємо ознаки для тестових зображень...")
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        feats = features_extractor(images)
        feats = torch.flatten(feats, 1)

        test_features.append(feats.cpu().numpy())
        test_labels.append(labels.numpy())

X_test = np.concatenate(test_features, axis=0)
y_test = np.concatenate(test_labels, axis=0)

print("Форма масиву X_test:", X_test.shape)
print("Форма масиву y_test:", y_test.shape)

print("\nОбчислюємо cosine similarity та класифікуємо тестові зображення...")

similarity_matrix = cosine_similarity(X_test, X_train)

predicted_labels = []
num_neighbors = 5

for similarities in similarity_matrix:
    top_k_indices = np.argsort(similarities)[-num_neighbors:]
    top_k_classes = y_train[top_k_indices]

    predicted_class = np.bincount(top_k_classes).argmax()
    predicted_labels.append(predicted_class)

predicted_labels = np.array(predicted_labels)

print("\n--- Classification Report ---")
print(classification_report(y_test, predicted_labels, target_names=train_dataset.classes, zero_division=0))

print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, predicted_labels))

Витягуємо ознаки для тестових зображень...
Форма масиву X_test: (265, 1280)
Форма масиву y_test: (265,)

Обчислюємо cosine similarity та класифікуємо тестові зображення...

--- Classification Report ---
                   precision    recall  f1-score   support

     ace of clubs       0.38      0.60      0.46         5
  ace of diamonds       0.50      0.80      0.62         5
    ace of hearts       0.21      0.60      0.32         5
    ace of spades       0.71      1.00      0.83         5
   eight of clubs       0.75      0.60      0.67         5
eight of diamonds       0.12      0.20      0.15         5
  eight of hearts       0.27      0.80      0.40         5
  eight of spades       0.40      0.40      0.40         5
    five of clubs       0.10      0.20      0.13         5
 five of diamonds       0.25      0.60      0.35         5
   five of hearts       0.00      0.00      0.00         5
   five of spades       0.17      0.20      0.18         5
    four of clubs       0.00 